# Non-Standard Dataset Extraction

Extracts a sample of ~40k loans per year (10k per quarter) from the Freddie Mac
Non-Standard Dataset (ARM, interest-only, limited documentation loans).

These are the loan types excluded from the Standard Dataset for not meeting
Credit Risk Transfer eligibility criteria — and are believed to be the riskier
loans that drove the 2008 crisis.

Output: a single Parquet file with origination data + default target,
ready to be merged with the Standard Dataset.

In [1]:
import zipfile
import io
import pandas as pd
from pathlib import Path

In [2]:
# Configuration
NSD_FOLDER = Path('../data/01_raw/non_std_historical_data')  # adjust to your actual path
YEARS = list(range(2000, 2009))
QUARTERS = ['Q1', 'Q2', 'Q3', 'Q4']
SAMPLE_PER_QUARTER = 10000  # 10k per quarter -> ~40k per year
RANDOM_STATE = 42

# NSD origination files have 31 columns (missing mi_cancellation_indicator
# compared to the Standard Dataset's 32 columns)
ORIGINATION_COLUMNS_NSD = [
    'credit_score',
    'first_payment_date',
    'first_time_homebuyer_flag',
    'maturity_date',
    'msa',
    'mi_percentage',
    'number_of_units',
    'occupancy_status',
    'original_cltv',
    'original_dti',
    'original_upb',
    'original_ltv',
    'original_interest_rate',
    'channel',
    'prepayment_penalty_flag',
    'amortization_type',
    'property_state',
    'property_type',
    'postal_code',
    'loan_sequence_number',
    'loan_purpose',
    'original_loan_term',
    'number_of_borrowers',
    'seller_name',
    'servicer_name',
    'super_conforming_flag',
    'pre_relief_refinance_loan_sequence_number',
    'special_eligibility_program',
    'relief_refinance_indicator',
    'property_valuation_method',
    'interest_only_indicator',
    'mi_cancellation_indicator'
]

DEFAULT_CODES = ['02', '03', '09']

## Helper functions

Files are nested: `historical_data_excl_{year}.zip` contains
`historical_data_excl_{year}{quarter}.zip`, which in turn contains the
actual `.txt` data files. We read everything in-memory without extracting
to disk.

In [3]:
def read_nsd_quarter_origination(year: int, quarter: str, sample_size: int) -> pd.DataFrame:
    """Read and sample origination data for a single NSD quarter.

    Args:
        year: origination year.
        quarter: quarter string, e.g. 'Q1'.
        sample_size: number of loans to randomly sample.
    Returns:
        Sampled origination DataFrame with named columns, or empty
        DataFrame if the file is missing or unreadable.
    """
    outer_zip_path = NSD_FOLDER / f'historical_data_excl_{year}.zip'

    if not outer_zip_path.exists():
        print(f'  [skip] {year}{quarter}: outer zip not found')
        return pd.DataFrame()

    try:
        with zipfile.ZipFile(outer_zip_path) as outer_zip:
            inner_zip_name = f'historical_data_excl_{year}{quarter}.zip'

            with outer_zip.open(inner_zip_name) as inner_zip_bytes:
                inner_zip_data = io.BytesIO(inner_zip_bytes.read())

                with zipfile.ZipFile(inner_zip_data) as inner_zip:
                    txt_name = f'historical_data_excl_{year}{quarter}.txt'

                    with inner_zip.open(txt_name) as f:
                        df = pd.read_csv(
                            f,
                            sep='|',
                            header=None,
                            names=ORIGINATION_COLUMNS_NSD,
                            low_memory=False,
                        )

        if len(df) > sample_size:
            df = df.sample(n=sample_size, random_state=RANDOM_STATE)

        print(f'  [ok]   {year}{quarter}: {len(df):,} loans sampled')
        return df

    except KeyError:
        print(f'  [skip] {year}{quarter}: inner file not found in archive')
        return pd.DataFrame()
    except Exception as e:
        print(f'  [error] {year}{quarter}: {e}')
        return pd.DataFrame()

In [17]:
def read_nsd_quarter_performance(year: int, quarter: str, loan_ids: set) -> pd.DataFrame:
    """Read performance data for a single NSD quarter, filtered to the
    sampled loan_sequence_numbers only (to keep memory usage low).

    The performance file's zero_balance_code is in column index 8
    (0-indexed), matching the Standard Dataset's PERFORMANCE_COLUMNS layout.

    Args:
        year: origination year.
        quarter: quarter string, e.g. 'Q1'.
        loan_ids: set of loan_sequence_number values to keep.
    Returns:
        DataFrame with loan_sequence_number and zero_balance_code only.
    """
    outer_zip_path = NSD_FOLDER / f'historical_data_excl_{year}.zip'

    if not outer_zip_path.exists():
        return pd.DataFrame()

    try:
        with zipfile.ZipFile(outer_zip_path) as outer_zip:
            inner_zip_name = f'historical_data_excl_{year}{quarter}.zip'

            with outer_zip.open(inner_zip_name) as inner_zip_bytes:
                inner_zip_data = io.BytesIO(inner_zip_bytes.read())

                with zipfile.ZipFile(inner_zip_data) as inner_zip:
                    txt_name = f'historical_data_excl_time_{year}{quarter}.txt'

                    with inner_zip.open(txt_name) as f:
                        # Only load columns 0 (loan_sequence_number) and 8 (zero_balance_code)
                        df = pd.read_csv(
                            f,
                            sep='|',
                            header=None,
                            usecols=[0, 8],
                            names=['loan_sequence_number', 'zero_balance_code'],
                            low_memory=False,
                        )

        df = df[df['loan_sequence_number'].isin(loan_ids)]
        
        # Marca se a linha tem um código relevante de zero balance
        df['is_default_code'] = df['zero_balance_code'].notna()
        
        # Por empréstimo: mantém a primeira linha com código, OU a última linha se nunca teve código
        df = (
            df.sort_values(['loan_sequence_number', 'is_default_code'], ascending=[True, False])
            .groupby('loan_sequence_number', as_index=False)
            .first()
        )
        return df

    except KeyError:
        return pd.DataFrame()
    except Exception as e:
        print(f'  [error] performance {year}{quarter}: {e}')
        return pd.DataFrame()

## Extract origination samples for all years

In [5]:
origination_frames = []

for year in YEARS:
    print(f'Year {year}:')
    for quarter in QUARTERS:
        df_q = read_nsd_quarter_origination(year, quarter, SAMPLE_PER_QUARTER)
        if not df_q.empty:
            df_q['year'] = year
            origination_frames.append(df_q)

nsd_origination = pd.concat(origination_frames, ignore_index=True)
print(f'\nTotal NSD origination sample: {len(nsd_origination):,} loans')
nsd_origination.head()

Year 2000:
  [ok]   2000Q1: 10,000 loans sampled
  [ok]   2000Q2: 10,000 loans sampled
  [ok]   2000Q3: 10,000 loans sampled
  [ok]   2000Q4: 10,000 loans sampled
Year 2001:
  [ok]   2001Q1: 10,000 loans sampled
  [ok]   2001Q2: 10,000 loans sampled
  [ok]   2001Q3: 10,000 loans sampled
  [ok]   2001Q4: 10,000 loans sampled
Year 2002:
  [ok]   2002Q1: 10,000 loans sampled
  [ok]   2002Q2: 10,000 loans sampled
  [ok]   2002Q3: 10,000 loans sampled
  [ok]   2002Q4: 10,000 loans sampled
Year 2003:
  [ok]   2003Q1: 10,000 loans sampled
  [ok]   2003Q2: 10,000 loans sampled
  [ok]   2003Q3: 10,000 loans sampled
  [ok]   2003Q4: 10,000 loans sampled
Year 2004:
  [ok]   2004Q1: 10,000 loans sampled
  [ok]   2004Q2: 10,000 loans sampled
  [ok]   2004Q3: 10,000 loans sampled
  [ok]   2004Q4: 10,000 loans sampled
Year 2005:
  [ok]   2005Q1: 10,000 loans sampled
  [ok]   2005Q2: 10,000 loans sampled
  [ok]   2005Q3: 10,000 loans sampled
  [ok]   2005Q4: 10,000 loans sampled
Year 2006:
  [ok]   20

,credit_score,first_payment_date,first_time_homebuyer_flag,maturity_date,msa,mi_percentage,number_of_units,occupancy_status,original_cltv,original_dti,...,seller_name,servicer_name,super_conforming_flag,pre_relief_refinance_loan_sequence_number,special_eligibility_program,relief_refinance_indicator,property_valuation_method,interest_only_indicator,mi_cancellation_indicator,year
0,648,200003,N,203002,31084.0,0,1,P,75,29,...,Other sellers,CHASE MANHATTAN MORTGAGE CORPORATION,NaN,NaN,9,NaN,9,N,NaN,2000
1,693,200005,N,203004,47894.0,25,1,I,90,19,...,FIRST HORIZON HOME LOAN CORPORATION,FIRST HORIZON HOME LOAN CORPORATION,NaN,NaN,9,NaN,9,N,NaN,2000
2,661,200005,Y,203004,22744.0,30,1,P,95,31,...,FIRST HORIZON HOME LOAN CORPORATION,FIRST HORIZON HOME LOAN CORPORATION,NaN,NaN,9,NaN,9,N,NaN,2000
3,655,200004,Y,203003,NaN,25,1,P,95,34,...,Other sellers,"BANK OF AMERICA, N.A.",NaN,NaN,9,NaN,9,N,NaN,2000
4,747,200004,9,201503,NaN,0,1,P,91,999,...,"BANK OF AMERICA, N.A.","BANK OF AMERICA, N.A.",NaN,NaN,9,NaN,9,N,NaN,2000


## Extract performance data for the sampled loans

We only fetch performance records for the loan_sequence_numbers we already
sampled, to avoid loading the full multi-GB performance files.

In [18]:
sampled_loan_ids = set(nsd_origination['loan_sequence_number'])

performance_frames = []

for year in YEARS:
    print(f'Performance — Year {year}:')
    for quarter in QUARTERS:
        df_perf = read_nsd_quarter_performance(year, quarter, sampled_loan_ids)
        if not df_perf.empty:
            performance_frames.append(df_perf)
            print(f'  [ok]   {year}{quarter}: {len(df_perf):,} performance records')

nsd_performance = pd.concat(performance_frames, ignore_index=True)
print(f'\nTotal NSD performance records: {len(nsd_performance):,}')

Performance — Year 2000:
  [ok]   2000Q1: 10,000 performance records
  [ok]   2000Q2: 10,000 performance records
  [ok]   2000Q3: 10,000 performance records
  [ok]   2000Q4: 10,000 performance records
Performance — Year 2001:
  [ok]   2001Q1: 10,000 performance records
  [ok]   2001Q2: 10,000 performance records
  [ok]   2001Q3: 10,000 performance records
  [ok]   2001Q4: 10,000 performance records
Performance — Year 2002:
  [ok]   2002Q1: 10,000 performance records
  [ok]   2002Q2: 9,999 performance records
  [ok]   2002Q3: 10,000 performance records
  [ok]   2002Q4: 10,000 performance records
Performance — Year 2003:
  [ok]   2003Q1: 10,000 performance records
  [ok]   2003Q2: 10,000 performance records
  [ok]   2003Q3: 9,999 performance records
  [ok]   2003Q4: 10,000 performance records
Performance — Year 2004:
  [ok]   2004Q1: 10,000 performance records
  [ok]   2004Q2: 10,000 performance records
  [ok]   2004Q3: 10,000 performance records
  [ok]   2004Q4: 10,000 performance recor

## Build the default target

A loan is considered defaulted if its zero_balance_code is ever 02, 03, or 09.

In [19]:
nsd_performance['zero_balance_code_clean'] = (
    pd.to_numeric(nsd_performance['zero_balance_code'], errors='coerce')
    .astype('Int64')
    .astype('string')
    .str.zfill(2)
)

nsd_target = (
    nsd_performance.groupby('loan_sequence_number')['zero_balance_code_clean']
    .apply(lambda codes: int(codes.isin(DEFAULT_CODES).any()))
    .reset_index()
    .rename(columns={'zero_balance_code_clean': 'default'})
)

print(f'Loans with target: {len(nsd_target):,}')
print(f'Default rate: {nsd_target["default"].mean()*100:.2f}%')

Loans with target: 358,630
Default rate: 8.34%


## Join origination with target

In [20]:
nsd_full = nsd_origination.merge(nsd_target, on='loan_sequence_number', how='inner')
nsd_full['default'] = nsd_full['default'].astype(int)
nsd_full['source'] = 'non_standard'

print(f'Final NSD dataset: {len(nsd_full):,} loans')
print(f'\nDefault rate by year:')
print(nsd_full.groupby('year')['default'].agg(['count', 'mean']))

Final NSD dataset: 358,630 loans

Default rate by year:
      count      mean
year                 
2000  40000  0.023200
2001  40000  0.024500
2002  39999  0.023051
2003  39999  0.030926
2004  40000  0.051225
2005  39998  0.115306
2006  39998  0.187059
2007  40000  0.219100
2008  38636  0.076405


## Quick comparison: ARM vs FRM default rates

In [21]:
print('Amortization type distribution:')
print(nsd_full['amortization_type'].value_counts())
print()
print('Default rate by amortization type:')
print(nsd_full.groupby('amortization_type')['default'].mean())

Amortization type distribution:
amortization_type
FRM    229371
ARM    129259
Name: count, dtype: int64

Default rate by amortization type:
amortization_type
ARM    0.095289
FRM    0.076771
Name: default, dtype: float64


## Save to Parquet for later use

In [24]:
# Força conversão de todas as colunas que deveriam ser numéricas
numeric_cols = [
    'credit_score', 'first_payment_date', 'maturity_date', 'msa',
    'mi_percentage', 'number_of_units', 'original_cltv', 'original_dti',
    'original_upb', 'original_ltv', 'original_interest_rate',
    'postal_code', 'original_loan_term', 'number_of_borrowers',
]

for col in numeric_cols:
    if col in nsd_full.columns:
        nsd_full[col] = pd.to_numeric(nsd_full[col], errors='coerce')

nsd_full.to_parquet(output_path, index=False)
print(f'Saved to {output_path}')
print(f'Shape: {nsd_full.shape}')

Saved to ..\data\02_intermediate\nsd_sample.parquet
Shape: (358630, 35)


In [25]:
output_path = Path('../data/02_intermediate/nsd_sample.parquet')
output_path.parent.mkdir(parents=True, exist_ok=True)
nsd_full.to_parquet(output_path, index=False)
print(f'Saved to {output_path}')
print(f'Shape: {nsd_full.shape}')

Saved to ..\data\02_intermediate\nsd_sample.parquet
Shape: (358630, 35)


In [1]:
import pandas as pd
df = pd.read_parquet('../data/02_intermediate/model_input_data_all.parquet')
print(df.shape)
print(df['year'].value_counts().sort_index())
print(f"Default rate: {df['default'].mean()*100:.2f}%")

(808630, 34)
year
2000    90000
2001    90000
2002    89999
2003    89999
2004    90000
2005    89998
2006    89998
2007    90000
2008    88636
Name: count, dtype: int64
Default rate: 5.78%


In [2]:
df.columns

Index(['credit_score', 'first_payment_date', 'first_time_homebuyer_flag',
       'maturity_date', 'msa', 'mi_percentage', 'number_of_units',
       'occupancy_status', 'original_cltv', 'original_dti', 'original_upb',
       'original_ltv', 'original_interest_rate', 'channel',
       'prepayment_penalty_flag', 'amortization_type', 'property_state',
       'property_type', 'postal_code', 'loan_sequence_number', 'loan_purpose',
       'original_loan_term', 'number_of_borrowers', 'seller_name',
       'servicer_name', 'super_conforming_flag',
       'pre_relief_refinance_loan_sequence_number',
       'special_eligibility_program', 'relief_refinance_indicator',
       'property_valuation_method', 'interest_only_indicator',
       'mi_cancellation_indicator', 'default', 'year'],
      dtype='object')

In [1]:
import pandas as pd
df = pd.read_parquet('../data/03_primary/model_input_cleaned.parquet')
print(df.shape)
print(df['default'].mean())

(808630, 28)
0.05781012329495567


In [ ]:
import json
with open('../data/08_reporting/model_training_metrics.json') as f:
    metrics = json.load(f)
print(json.dumps(metrics, indent=2))

FileNotFoundError: [Errno 2] No such file or directory: 'data/08_reporting/model_training_metrics.json'

In [3]:
import pandas as pd
df = pd.read_parquet('../data/03_primary/model_input_cleaned.parquet')
print(df.shape)
print(df['year'].value_counts().sort_index())
print(f"Default rate: {df['default'].mean()*100:.2f}%")

(808630, 28)
year
2000    90000
2001    90000
2002    89999
2003    89999
2004    90000
2005    89998
2006    89998
2007    90000
2008    88636
Name: count, dtype: int64
Default rate: 5.78%


In [2]:
import pandas as pd

df = pd.read_parquet('../data/03_primary/model_input_cleaned.parquet')
train = df[df['year'] <= 2002]

print(f"Total de empréstimos no treino: {len(train):,}")
print(f"Taxa de default no treino: {train['default'].mean()*100:.2f}%")
print()
print("Por ano:")
print(train.groupby('year')['default'].agg(['count', 'mean']))

Total de empréstimos no treino: 269,999
Taxa de default no treino: 1.74%

Por ano:
      count      mean
year                 
2000  90000  0.017456
2001  90000  0.017456
2002  89999  0.017411


In [ ]:
print("Default rate por ano (todo o dataset):")
print(df.groupby('year')['default'].agg(['count', 'mean']).rename(columns={'mean': 'default_rate'}))

print()
print("Resumo treino vs teste:")
train = df[df['year'] <= 2002]
test = df[df['year'] >= 2003]
print(f"Treino (2000-2002): {len(train):,} empréstimos, {train['default'].mean()*100:.2f}% default")
print(f"Teste  (2003-2008): {len(test):,} empréstimos, {test['default'].mean()*100:.2f}% default")

Default rate por ano (todo o dataset):
      count  default_rate
year                     
2000  90000      0.017456
2001  90000      0.017456
2002  89999      0.017411
2003  89999      0.021500
2004  90000      0.038900
2005  89998      0.081124
2006  89998      0.125725
2007  90000      0.144700
2008  88636      0.055993

Resumo treino vs teste:
Treino (2000-2002): 269,999 empréstimos, 1.74% default
Teste  (2003-2008): 538,631 empréstimos, 7.80% default
